
# Criação do dataset para o modelo

Objetivo: Criar, para cada data mensal de referência, quais clientes estavam elegíveis e qual foi a target observada nos 30 dias seguintes.

In [0]:
#---- Libs

from pyspark.sql.functions import *
from pyspark.sql.window import Window

#---- Tabelas

SOURCE_TABLE = 'workspace.silver_layer.online_retail_transactions'
TARGET_TABLE = 'workspace.gold_layer.customer_reactivation_modeling_base'

#---- Parâmetros

INACTIVITY_DAYS = 60
TARGET_WINDOW_DAYS = 30
REFERENCE_START_DATE = "2010-03-08"
REFERENCE_END_DATE = "2011-11-08"


## Leitura dos dados

In [0]:
df = spark.sql(
    f'''
    select * from {SOURCE_TABLE}
    '''
)

df.display()


## Tratamento dos dados


### Criação dos campos: 

- is_cancelled_v2
- is_test
- is_discount
- total_amount
- is_zero_price

In [0]:
df_with_cols = (
    df
    .withColumn(
        "is_cancelled",
        col("invoice").startswith("C")
    )
    .withColumn(
        "is_test",
        col("stock_code").startswith("TEST")
    )
    .withColumn(
        "is_discount",
        col("description") == lit('Discount')
    )
    .withColumn(
        "total_amount",
        col('quantity') * col('price')
    )
    .withColumn(
        'is_zero_price', 
        col('price') == lit(0)
    )
    .withColumn(
        'is_zero_quantity', 
        col('quantity') == lit(0)
    )
    .withColumn(
        'purchase_date', 
        to_date('invoice_date')
    )
)

df_with_cols.display()


### Filtrando linhas que não serão úteis para o modelo

In [0]:
df_with_filters = (
    df_with_cols
    .filter(col('is_cancelled') == False)
    .filter(col('is_test') == False)
    .filter(col('is_zero_price') == False)
    .filter(col('is_zero_quantity') == False)
)

df_with_filters.display()


## Criação das datas mensais de referência


In [0]:
df_reference_dates = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(REFERENCE_START_DATE).cast("date"),
                lit(REFERENCE_END_DATE).cast("date"),
                expr("INTERVAL 1 MONTH"),
            )
        ).alias("reference_date")
    )
    .orderBy("reference_date")
)

df_reference_dates.display()


## Criação da população elegível

In [0]:
df_customer_history = (
    df_with_filters
    # Como existem somente poucas datas mensais,
    # podemos enviar esse pequeno dataframe
    # para todos os nós do Spark
    .crossJoin(broadcast(df_reference_dates))
    # Utiliza apenas compras realizadas
    # até a respectiva data de referência
    .filter(col("purchase_date") <= col("reference_date"))
)

df_customer_history.display()

In [0]:
df_customer_last_purchase = (
    df_customer_history.groupBy("customer_id", "reference_date")
    .agg(max("purchase_date").alias("last_purchase_date"))
    .withColumn(
        "inactive_days", datediff(col("reference_date"), col("last_purchase_date"))
    )
)

df_customer_last_purchase.display()

In [0]:
df_eligible_population = (
    df_customer_last_purchase
    .filter(
        col("inactive_days")
        >=
        lit(INACTIVITY_DAYS)
    )
)

df_eligible_population.display()


## Criação da variável resposta

In [0]:
df_reactivated = (
    df_eligible_population.alias("population")
    .join(
        df_with_filters.select("customer_id", "purchase_date")
        .distinct()
        .alias("purchases"),
        on=(
            (col("population.customer_id") == col("purchases.customer_id"))
            & (col("purchases.purchase_date") > col("population.reference_date"))
            & (
                col("purchases.purchase_date")
                <= date_add(col("population.reference_date"), TARGET_WINDOW_DAYS)
            )
        ),
        how="inner",
    )
    .select(
        col("population.customer_id").alias("customer_id"),
        col("population.reference_date").alias("reference_date"),
    )
    .distinct()
    .withColumn("target_reactivated_30d", lit(1))
)

df_reactivated.display()

In [0]:
df_population_with_target = df_eligible_population.join(
    df_reactivated, on=["customer_id", "reference_date"], how="left"
).fillna({"target_reactivated_30d": 0})

df_population_with_target.display()

In [0]:
df_target_distribution = (
    df_population_with_target.groupBy("reference_date")
    .agg(
        count("*").alias("eligible_customers"),
        sum("target_reactivated_30d").alias("reactivated_customers"),
    )
    .withColumn(
        "reactivation_rate",
        round((col("reactivated_customers") / col("eligible_customers")) * 100, 2),
    )
    .orderBy("reference_date")
)

df_target_distribution.display()


## Criação das features

In [0]:
df_feature_history = (
    df_population_with_target
    # Precisamos somente das chaves
    # para construir o histórico
    .select("customer_id", "reference_date")
    .alias("population")
    .join(
        df_with_filters.alias("transactions"),
        on=(
            (col("population.customer_id") == col("transactions.customer_id"))
            & (col("transactions.purchase_date") <= col("population.reference_date"))
        ),
        how="inner",
    )
    .select(
        col("population.customer_id").alias("customer_id"),
        col("population.reference_date").alias("reference_date"),
        col("transactions.invoice").alias("invoice"),
        col("transactions.stock_code").alias("stock_code"),
        col("transactions.quantity").alias("quantity"),
        col("transactions.total_amount").alias("total_amount"),
        col("transactions.purchase_date").alias("purchase_date"),
        col("transactions.country").alias("country")
    )
)

df_feature_history.display()

In [0]:
df_customer_features = (
    df_feature_history.groupBy("customer_id", "reference_date")
    .agg(
        # Data da primeira compra válida
        min("purchase_date").alias("first_purchase_date"),
        # Quantidade de pedidos diferentes
        countDistinct("invoice").alias("number_orders"),
        # Quantidade total de itens comprados
        sum("quantity").alias("total_items"),
        # Valor histórico total comprado
        sum("total_amount").alias("total_spent"),
        # Quantidade de produtos diferentes
        countDistinct("stock_code").alias("unique_products"),
        # Quantidade de países diferentes
        countDistinct("country").alias("distinct_countries"),
    )
    # Quantidade de dias desde
    # a primeira compra
    .withColumn(
        "customer_tenure_days",
        datediff(col("reference_date"), col("first_purchase_date")),
    )
    # Valor médio por pedido
    .withColumn(
        "average_ticket",
        when(col("number_orders") > 0, col("total_spent") / col("number_orders")),
    )
    # Valor médio acumulado
    # por país diferente
    .withColumn(
        "average_amount_per_country",
        when(
            col("distinct_countries") > 0,
            col("total_spent") / col("distinct_countries"),
        ),
    )
    # Valor médio acumulado
    # por produto diferente
    .withColumn(
        "average_amount_per_stock_code",
        when(col("unique_products") > 0, col("total_spent") / col("unique_products")),
    )
)

df_customer_features.display()

In [0]:
df_modeling_base = df_population_with_target.join(
    df_customer_features, on=["customer_id", "reference_date"], how="left"
)

df_modeling_base.display()


## Salvar a tabela no catálogo gold

In [0]:
(
    df_modeling_base
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold_layer.customer_reactivation_modeling"
    )
)